<a href="https://colab.research.google.com/github/RAseng77/AIFFEL_QUEST_RS/blob/master/Exploration/Ex09/Transformer_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.3 MB/s eta 0:00:00


In [ ]:
# 1. 폴더 생성 (터미널 명령어는 !를 붙임)
!mkdir -p ~/work/transformer_chatbot/data/

# 2. 경로 이동 (중요: cd는 !가 아니라 %를 써야 영구적으로 이동됨)
%cd ~/work/transformer_chatbot/data/

# 3. 파일 다운로드 (! 사용)
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv

# 4. 확인 (잘 받아졌는지 파일 목록 출력)
!ls -l

/home/jovyan/work/transformer_chatbot/data
--2026-02-06 01:18:18--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-02-06 01:18:19--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.1’

ChatbotData.csv.1   100%[===================>] 868.99K  4.92MB/s    in 0.2s    

2026-02-06 01:18:19 (4.92 MB/s) - ‘ChatbotData.csv.1’ saved [889842/88984

In [ ]:
import pandas as pd

data = pd.read_csv('ChatbotData.csv')

print(f"데이터 개수 {len(data)}")
print(data.head())

데이터 개수 11823
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


In [ ]:
MAX_SAMPLES = 50000

In [ ]:
import re

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()

    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)

    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    sentence = sentence.strip()
    return sentence

In [ ]:
print("전처리 전:", data['Q'][0])
print("전처리 후:", preprocess_sentence(data['Q'][0]))

전처리 전: 12시 땡!
전처리 후: 12시 땡 !


In [ ]:
corpus = []
for sentence in data['Q']:
    corpus.append(preprocess_sentence(sentence))
for sentence in data['A']:
    corpus.append(preprocess_sentence(sentence))

# text 파일로 저장
with open('chatbot_corpus.txt', 'w', encoding='utf-8') as f:
    for sentence in corpus:
        f.write(sentence + '\n')

print("코퍼스 저장 완료: chatbot_corpus.txt")

코퍼스 저장 완료: chatbot_corpus.txt


In [ ]:
import sentencepiece as spm

# 파라미터 설정
input_file = 'chatbot_corpus.txt'
vocab_size = 8000
model_name = 'korean_spm'
model_type = 'unigram'
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

input_argument = '--input={} --model_prefix={} --vocab_size={} --user_defined_symbols={} --model_type={} --pad_id={} --bos_id={} --eos_id={} --unk_id={}'.format(
    input_file, model_name, vocab_size, 'some,symbols', model_type, pad_id, bos_id, eos_id, unk_id
)

spm.SentencePieceTrainer.Train(input_argument)

print("SentencePiece 학습 완료!")

SentencePiece 학습 완료!


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=chatbot_corpus.txt --model_prefix=korean_spm --vocab_size=8000 --user_defined_symbols=some,symbols --model_type=unigram --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot_corpus.txt
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: some
  user_defined_symbols: symbols
  required_chars: 
  byte_fallback: 0
  vocabulary_output_pi

In [ ]:
# 토크나이저 로드
sp = spm.SentencePieceProcessor()
sp.Load(f'{model_name}.model')

True

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 40

class ChatbotDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = df
        self.max_len = max_len
        self.question = df['Q'].tolist()
        self.answers = df['A'].tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 전처리
        q_text = preprocess_sentence(self.question[idx])
        a_text = preprocess_sentence(self.answers[idx])

        # 토크나이징
        q_ids = [self.tokenizer.bos_id()] + self.tokenizer.EncodeAsIds(q_text) + [self.tokenizer.eos_id()]
        a_ids = [self.tokenizer.bos_id()] + self.tokenizer.EncodeAsIds(a_text) + [self.tokenizer.eos_id()]

        # 패딩
        if len(q_ids) < self.max_len:
            q_ids = q_ids + [self.tokenizer.pad_id()] * (self.max_len - len(q_ids))
        else:
            q_ids = q_ids[:self.max_len]

        if len(a_ids) < self.max_len:
            a_ids = a_ids + [self.tokenizer.pad_id()] * (self.max_len - len(a_ids))
        else:
            a_ids = a_ids[:self.max_len]

        return torch.tensor(q_ids), torch.tensor(a_ids)


In [ ]:
dataset = ChatbotDataset(data, sp, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
sample_q, sample_a = next(iter(dataloader))

print("Q 배치 크기: ",sample_q.shape)
print("A 배치 크기: ",sample_a.shape)

Q 배치 크기:  torch.Size([64, 40])
A 배치 크기:  torch.Size([64, 40])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 마스킹
def create_padding_mask(seq):
    seq = torch.eq(seq, 0).float()

    # [수정] seq 변수가 빠져있어서 추가했습니다.
    return seq[:, None, None, :]

def create_look_ahead_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def create_masks(src, trg):
    "src: 인코더 입력, trg: 디코더 입력"

    # 인코더용 패딩마스크
    src_mask = create_padding_mask(src)

    # 디코더용 패딩 마스크
    trg_padding_mask = create_padding_mask(trg)

    # 디코더용 멀티헤드 어텐션 가리기
    look_ahead_mask = create_look_ahead_mask(trg.size(1)).type_as(trg_padding_mask)

    # 디코더 최종 마스크
    dec_mask = torch.max(trg_padding_mask, look_ahead_mask)

    return src_mask, dec_mask

## Positional Encoding, Multi-Head Attention, FFN

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)

        self.dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0 ,2, 1, 3)

    def forward(self, v, k, q, mask):
        batch_size = q.size(0)

        # [수정] wk에는 k, wv에는 v를 넣도록 수정했습니다. (기존엔 모두 q였음)
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        matmul_qk = torch.matmul(q, k.transpose(-1, -2))

        dk = k.size()[-1]
        scaled_attention_logits = matmul_qk / math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = F.softmax(scaled_attention_logits, dim=-1)

        output = torch.matmul(attention_weights, v)

        output = output.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)

        return self.dense(output), attention_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, dff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, dff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dff, d_model)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

## 인코더 & 디코더 레이어
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, dff, dropout)

        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output, _ = self.mha(x, x, x, mask)
        out1 = self.layernorm1(x + self.dropout1(attn_output))

        ffn_output = self.ffn(out1)
        out2 = self.layernorm2(out1 + self.dropout2(ffn_output))

        return out2

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        # Masked self Attention
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        # Encoder-Decoder Attention
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, dff, dropout)

        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.layernorm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):

        # Masked Self Attention
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(x + self.dropout1(attn1))

        # Encoder-Decoder Attention
        attn2, attn_weights_block2 = self.mha2(enc_output, enc_output, out1, padding_mask)
        # [수정] dropout -> dropout2
        out2 = self.layernorm2(out1 + self.dropout2(attn2))

        # Feed Forward
        ffn_output = self.ffn(out2)
        # [수정] dropout -> dropout3
        out3 = self.layernorm3(out2 + self.dropout3(ffn_output))

        return out3, attn_weights_block1, attn_weights_block2


## 트랜스포머 모델
class Transformer(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, target_vocab_size, pe_input, pe_target, dropout=0.1):
        super(Transformer, self).__init__()

        self.d_model = d_model

        # 인코더
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.encoder_pos_encoding = PositionalEncoding(d_model, pe_input)

        # [수정] num_layeres -> num_layers (오타 수정)
        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, dff, dropout) for _ in range(num_layers)]
        )
        self.encoder_dropout = nn.Dropout(dropout)

        # 디코더
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.decoder_pos_encoding = PositionalEncoding(d_model, pe_target)
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, dff, dropout) for _ in range(num_layers)]
        )
        self.decoder_dropout = nn.Dropout(dropout)

        # 출력
        self.final_layer = nn.Linear(d_model, target_vocab_size)

    def encode(self, inp, training, mask):
        x = self.encoder_embedding(inp) * math.sqrt(self.d_model)
        x = self.encoder_pos_encoding(x)
        x = self.encoder_dropout(x)

        for layer in self.encoder_layers:
            x = layer(x, mask)
        return x

    def decode(self, tar, enc_output, training, look_ahead_mask, padding_mask):
        x = self.decoder_embedding(tar) * math.sqrt(self.d_model)
        x = self.decoder_pos_encoding(x)
        x = self.decoder_dropout(x)

        attention_weights = {}

        for i, layer in enumerate(self.decoder_layers):
            x, block1, block2 = layer(x, enc_output, look_ahead_mask, padding_mask)

            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2

        return x, attention_weights

    def forward(self, inp, tar, training=True):
        """
        inp: 인코더 입력
        tar: 디코더 입력
        """

        # 마스크 생성
        src_mask, dec_mask = create_masks(inp, tar)

        # 인코더 실행 -> 인코더 출력 생성
        enc_output = self.encode(inp, training, src_mask)

        # 디코더 실행 -> 디코더 출력 생성
        dec_output, attention_weights = self.decode(tar, enc_output, training, dec_mask, src_mask)

        # 최종 출력층 통과
        final_output = self.final_layer(dec_output)

        return final_output

In [ ]:
# 모델 생성
num_layers = 2
d_model = 256
num_heads = 8
dff = 512
dropout = 0.1
vocab_size = sp.GetPieceSize() # 8000

model = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    input_vocab_size=vocab_size,
    target_vocab_size=vocab_size,
    pe_input=1000,
    pe_target=1000,
    dropout=dropout
)

print("모델 생성 완료!")

모델 생성 완료!


In [ ]:
def evaluate(sentence):
    # 전처리
    sentence = preprocess_sentence(sentence)

    # 토크나이징 및 인코딩하기
    input_ids = [sp.bos_id()] + sp.EncodeAsIds(sentence) + [sp.eos_id()]

    # 텐서로 변환
    encoder_input = torch.tensor([input_ids])

    # 인코더 입력 패딩 마스크 생성
    src_mask = create_padding_mask(encoder_input)

    # 디코더 입력
    decoder_input = torch.tensor([[sp.bos_id()]])

    model.eval()

    with torch.no_grad():
        src_mask = create_padding_mask(encoder_input)
        enc_output = model.encode(encoder_input, training=False, mask=src_mask)

        for i in range(40):
            trg_padding_mask = create_padding_mask(decoder_input)
            look_ahead_mask = create_look_ahead_mask(decoder_input.size(1))
            combined_mask = torch.max(trg_padding_mask, look_ahead_mask)

            dec_output, attention_weights = model.decode(
                decoder_input, enc_output, training=False,
                look_ahead_mask = combined_mask, padding_mask=src_mask
            )

            predictions = model.final_layer(dec_output)
            prediction = predictions[:, -1, :]
            predicted_id = torch.argmax(prediction, dim=-1).item()

            if predicted_id == sp.eos_id():
                break


            decoder_input = torch.cat(
                [decoder_input, torch.tensor([[predicted_id]])], dim=-1
            )

    result_ids = decoder_input.squeeze().tolist()
    return sp.DecodeIds(result_ids[1:])

def predict(sentence):
    result = evaluate(sentence)
    return result

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

def get_transformer_schedule(d_model, warmup_steps):
    def lr_lambda(step):
        step = step + 1

        arg1 = step ** -0.5
        arg2 = step * (warmup_steps ** -1.5)

        return (d_model ** -0.5) * min(arg1, arg2)

    return lr_lambda

In [ ]:
warmup_steps = 4000
d_model = 256

criterion = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=get_transformer_schedule(d_model, warmup_steps)
)


In [ ]:
import os

if not os.path.exists('checkpoints'):
    os.makedirs('checkpoints')

EPOCHS = 10

for epoch in range(EPOCHS):
    total_loss = 0

    for batch, (questions, answers) in enumerate(dataloader):

        model.train()

        dec_inputs = answers[:, :-1]
        outputs = answers[:, 1:]

        optimizer.zero_grad()

        predictions = model(questions, dec_inputs)
        loss = criterion(predictions.view(-1, predictions.size(-1)), outputs.contiguous().view(-1))

        loss.backward()
        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        if batch % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1} | Batch {batch} | Loss: {loss.item():.4f} | LR: {current_lr:.8f}")

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} 완료 | 평균 Loss: {avg_loss:.4f}")

    print("\n [중간테스트]")
    try:
        test_samples =["오늘 날씨 어떻니?", "뭐하고 있어?", "너는 누구야"]

        for q in test_samples:
            print(f"Q: {q}")
            print(f"A: {predict(q)}")
            print("-" * 20)
    except Exception as e:
        print(f"테스트 중 에러 발생: {e}")


    save_path = f"checkpoints/transformer_epoch_{epoch+1}_loss_{avg_loss:.4f}.pt"

    # 모델의 가중치만 저장
    torch.save(model.state_dict(), save_path)
    print(f"모델 저장 완료: {save_path}\n" + "="*50)

Epoch 1 | Batch 0 | Loss: 5.7281 | LR: 0.00000049
Epoch 1 | Batch 100 | Loss: 5.2884 | LR: 0.00002520
Epoch 1 완료 | 평균 Loss: 5.4397

 [중간테스트]
Q: 오늘 날씨 어떻니?
A: 요 .
--------------------
Q: 뭐하고 있어?
A: 요 .
--------------------
Q: 너는 누구야
A: 요 .
--------------------
모델 저장 완료: checkpoints/transformer_epoch_1_loss_5.4397.pt
Epoch 2 | Batch 0 | Loss: 5.3716 | LR: 0.00004620
Epoch 2 | Batch 100 | Loss: 5.1469 | LR: 0.00007090
Epoch 2 완료 | 평균 Loss: 5.2232

 [중간테스트]
Q: 오늘 날씨 어떻니?
A: 해보세요 .
--------------------
Q: 뭐하고 있어?
A: 해보세요 .
--------------------
Q: 너는 누구야
A: 좋은 해보세요 .
--------------------
모델 저장 완료: checkpoints/transformer_epoch_2_loss_5.2232.pt
Epoch 3 | Batch 0 | Loss: 5.1223 | LR: 0.00009190
Epoch 3 | Batch 100 | Loss: 4.9414 | LR: 0.00011661
Epoch 3 완료 | 평균 Loss: 4.9359

 [중간테스트]
Q: 오늘 날씨 어떻니?
A: 해보세요 .
--------------------
Q: 뭐하고 있어?
A: 해보세요 .
--------------------
Q: 너는 누구야
A: 해보세요 .
--------------------
모델 저장 완료: checkpoints/transformer_epoch_3_loss_4.9359.pt
Epoch 4 | Batch 0 | Loss: 4.

In [ ]:
model = Transformer(
    num_layers=2, d_model=256, num_heads=8, dff=512,
    input_vocab_size=sp.GetPieceSize(), target_vocab_size=sp.GetPieceSize(),
    pe_input=1000, pe_target=1000, dropout=0.1
)

checkpoint_path = "checkpoints/transformer_epoch_10_loss_2.9953.pt"

try:
    model.load_state_dict(torch.load(checkpoint_path))
    print(f'모델 로드 성공 ({checkpoint_path})')
except Exception as e:
    print(f'파일 로드 실패: {e}')

모델 로드 성공 (checkpoints/transformer_epoch_10_loss_2.9953.pt)


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = LambdaLR(optimizer, lr_lambda=get_transformer_schedule(256, 4000))

In [ ]:
START_EPOCH = 11
TARGET_EPOCH = 100

patience_limit = 5 # Loss가 5번 연속으로 안 줄어들면 멈춤
patience_check = 0 # 몇번 정도 참았는지
min_loss = float('inf') # 최저 loss 기록

for epoch in range(START_EPOCH, TARGET_EPOCH + 1):
    total_loss = 0
    model.train()

    for batch, (questions, answers) in enumerate(dataloader):
        dec_inputs = answers[:, :-1]
        outputs = answers[:, 1:]

        optimizer.zero_grad()
        predictions = model(questions, dec_inputs)
        loss = criterion(predictions.view(-1, predictions.size(-1)), outputs.contiguous().view(-1))

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if batch % 100 == 0:
             print(f"Epoch {epoch} | Batch {batch} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} 완료 | 평균 Loss: {avg_loss:.4f}")

    save_path = f"checkpoints/transformer_epoch_{epoch}_loss_{avg_loss:.4f}.pt"
    torch.save(model.state_dict(), save_path)

    if avg_loss < min_loss:
        # Loss가 만약 줄어들었다면
        print(f"Loss ({min_loss:.4f} -> {avg_loss:.4f})")
        min_loss = avg_loss
        patience_check = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pt")
    else:
        patience_check += 1
        print(f"Loss가 줄지 않았어요 ({patience_check}/{patience_limit})")

        if patience_check >= patience_limit:
            print("!!STOP!!")
            break

    print("[중간 테스트]")
    try:
        test_q_list = ["오늘 날씨 어때?", "배고파", "사랑합니다."]
        for q in test_q_list:
            print(f"Q: {q} -> A: {predict(q)}")
    except:
        pass
    print("="*50)

Epoch 11 | Batch 0 | Loss: 2.5370
Epoch 11 | Batch 100 | Loss: 2.7322
Epoch 11 완료 | 평균 Loss: 2.6660
Loss (inf -> 2.6660)
[중간 테스트]
Q: 오늘 날씨 어때? -> A: 저는 위로봇입니다 .
Q: 배고파 -> A: 좋은 생각이에요 .
Q: 사랑합니다. -> A: 사랑의 예의가 없네요 .
Epoch 12 | Batch 0 | Loss: 2.5007
Epoch 12 | Batch 100 | Loss: 2.8063
Epoch 12 완료 | 평균 Loss: 2.6110
Loss (2.6660 -> 2.6110)
[중간 테스트]
Q: 오늘 날씨 어때? -> A: 맛있는 거 드세요 .
Q: 배고파 -> A: 좋은 생각이에요 .
Q: 사랑합니다. -> A: 사랑의 예의가 없네요 .
Epoch 13 | Batch 0 | Loss: 2.5793
Epoch 13 | Batch 100 | Loss: 2.6834
Epoch 13 완료 | 평균 Loss: 2.5482
Loss (2.6110 -> 2.5482)
[중간 테스트]
Q: 오늘 날씨 어때? -> A: 맛있는 거 드세요 .
Q: 배고파 -> A: 드세요 .
Q: 사랑합니다. -> A: 사랑의 예의가 없네요 .
Epoch 14 | Batch 0 | Loss: 2.4621
Epoch 14 | Batch 100 | Loss: 2.4399
Epoch 14 완료 | 평균 Loss: 2.4705
Loss (2.5482 -> 2.4705)
[중간 테스트]
Q: 오늘 날씨 어때? -> A: 저는 위로봇입니다 .
Q: 배고파 -> A: 드세요 .
Q: 사랑합니다. -> A: 사랑의 예의가 없네요 .
Epoch 15 | Batch 0 | Loss: 2.2440
Epoch 15 | Batch 100 | Loss: 2.4598
Epoch 15 완료 | 평균 Loss: 2.3632
Loss (2.4705 -> 2.3632)
[중간 테스트]
Q: 오늘 날씨

In [ ]:
def compute_accuracy(model, dataloader, pad_id):
    model.eval() # 평가 모드

    total_correct = 0
    total_count = 0

    print("정확도 계산 중...", end="")

    with torch.no_grad(): # 평가 땐 기울기 계산 X
        for questions, answers in dataloader:

            # 데이터 준비 (학습 때와 동일)
            dec_inputs = answers[:, :-1] # 입력: <s> 안녕
            targets = answers[:, 1:]     # 정답: 안녕 </s>

            # 모델 예측
            # predictions shape: (Batch, Seq_Len, Vocab_Size)
            predictions = model(questions, dec_inputs)

            # 가장 확률 높은 단어 선택 (Argmax)
            # predicted_ids shape: (Batch, Seq_Len)
            predicted_ids = torch.argmax(predictions, dim=-1)


            # 1. 패딩이 아닌 진짜 데이터만 골라내기 위한 마스크
            # targets가 pad_id(0)가 아닌 곳은 True, 0인 곳은 False
            non_pad_mask = targets.ne(pad_id)

            # 2. 정답과 예측이 같은지 확인 (True/False)
            correct_tensor = predicted_ids.eq(targets)

            # 3. 마스크 적용 (패딩이 아닌 곳 중에서 맞은 것만 남김)
            correct_masked = correct_tensor & non_pad_mask

            # 4. 개수 누적
            total_correct += correct_masked.sum().item() # 맞은 개수
            total_count += non_pad_mask.sum().item()     # 전체 유효 토큰 개수

            print(".", end="") # 진행상황 표시

    print(" 완료!")

    # 최종 정확도 반환
    return total_correct / total_count if total_count > 0 else 0


# 1. 가장 좋았던 모델 불러오기
best_model_path = "checkpoints/best_model.pt"

try:
    model.load_state_dict(torch.load(best_model_path))
    print(f"Best Model 로드 완료: {best_model_path}")
except:
    print("Best Model이 없어서 현재 로드된 모델로 측정합니다.")

# 2. 정확도 측정 실행
accuracy = compute_accuracy(model, dataloader, sp.pad_id())

print("=" * 50)
print(f"최종 모델 정확도(Accuracy): {accuracy * 100:.2f}%")
print("=" * 50)

Best Model 로드 완료: checkpoints/best_model.pt
정확도 계산 중............................................................................................................................................................................................ 완료!
최종 모델 정확도(Accuracy): 99.79%


### 회고
### 중간에 1에포크마다 테스트 단어 "오늘 날씨 어때?", "배고파", "사랑합니다." 와 같은 단어를 넣어서 실행해봤을떄
### 30에포크까지는 첫번째 단어와 두번째 단어는 질문 요지에 맞는 답별을 했지만 "사랑합니다"와 같은 감정적인 단어로 질문했을때는
### 질문요지에 맞는 답을 잘하지 못했음..
### 시간이 조금더 있었다면 감정이나 분위기를 알아야 답을 할수있는 텍스트를 선택해서 모델에 학습시켜보고싶습니다.